# 🧠 Classificação Multi-Tarefa de Emoções (Deep Learning vs ML Clássico)

Neste experimento acadêmico, atacamos o problema de **Multi-Label/Multi-Task Emotion Classification** utilizando o dataset `Google GoEmotions`.
O objetivo principal é avaliar a robustez de Redes Neurais Complexas (*Multi-gate Mixture of Experts - MMoE*) contra Algoritmos Clássicos de *Machine Learning* (Extra Trees, LightGBM, SVC) em cenários de **extrema esparsidade dimensional**.

### Tópicos Abordados:
- Extração de Bag-of-Words (TF-IDF) com Bigramas em alta dimensionalidade (15.000 features).
- Funções de Custo Otimizadas: Utilização de **Focal Loss Binária** para dados fortemente desbalanceados.
- Arquitetura PyTorch MMoE: Portões de especialistas dinâmicos compartilhados entre tarefas (Alegria, Tristeza, Raiva).
- Batalha Clássica: A prova matemática de que métodos de *Ensemble Randomizados* (Extra Trees) superam as Redes Neurais Profundas no domínio de matrizes 99% vazias.

In [ ]:
import os
import re
import warnings
import time

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import pandas as pd
import numpy as np
from pathlib import Path

import mlflow
import mlflow.pytorch

from sklearn.metrics import f1_score
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multioutput import MultiOutputClassifier
from sklearn.svm import LinearSVC
from lightgbm import LGBMClassifier
from sklearn.ensemble import ExtraTreesClassifier

warnings.filterwarnings('ignore')
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Setup MLFlow
mlflow.set_tracking_uri("https://dagshub.com/PedroM2626/experiments.mlflow")
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

## 1. Ingestão de Dados e Engenharia de Rótulos

Carregamos o dataset completo (43 mil amostras) e isolamos as três emoções fundamentais que queremos prever simultaneamente: **Alegria (Joy)**, **Tristeza (Sadness)** e **Raiva (Anger)**.

In [ ]:
print("[1] Baixando Dataset GoEmotions (Completo)...")
dataset = load_dataset("go_emotions", "simplified")

df = dataset['train'].to_pandas()
df_test = dataset['test'].to_pandas()

label_names = dataset['train'].features['labels'].feature.names
joy_idx = label_names.index('joy')
sadness_idx = label_names.index('sadness')
anger_idx = label_names.index('anger')

df['y_joy'] = df['labels'].apply(lambda x: 1 if joy_idx in x else 0)
df['y_sad'] = df['labels'].apply(lambda x: 1 if sadness_idx in x else 0)
df['y_ang'] = df['labels'].apply(lambda x: 1 if anger_idx in x else 0)

df_test['y_joy'] = df_test['labels'].apply(lambda x: 1 if joy_idx in x else 0)
df_test['y_sad'] = df_test['labels'].apply(lambda x: 1 if sadness_idx in x else 0)
df_test['y_ang'] = df_test['labels'].apply(lambda x: 1 if anger_idx in x else 0)

## 2. Feature Engineering: A Magia da Esparsidade

Em *Natural Language Processing* (NLP), um passo crucial é decidir como transformar o texto em matemática.
Ao invés de usar Embeddings densos (como DistilBERT), optamos por um método clássico: **TF-IDF**.

- **Limpeza de Ruído**: Removemos URLs e menções para evitar lixo estatístico.
- **Stop Words Mantidas**: Ao contrário da crença popular, manter palavras comuns (como "não", "muito") é vital para a classificação de emoções, pois elas alteram o contexto polar da sentença.
- **N-Grams**: Configuramos para extrair *Bigramas* (`ngram_range=(1,2)`). Dessa forma, capturamos não apenas "happy", mas também "not happy", aumentando a riqueza do vocabulário para um teto de 15.000 features.

### O Paradoxo da Matriz 99% Vazia
A matriz gerada possui 15.000 colunas. Contudo, um comentário típico do Reddit tem apenas cerca de 10 a 20 palavras. Isso significa que **99% da matriz é preenchida por ZEROS**. Mais tarde, veremos como essa característica dita qual algoritmo vence a competição.

In [ ]:
def clean_text(text):
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    return text.strip()

print("[2] Limpando e Extraindo Features com TF-IDF (CPU)...")
df['text'] = df['text'].apply(clean_text)
df_test['text'] = df_test['text'].apply(clean_text)

# Mantemos a matriz em formato Esparso (csr_matrix) nativo para economizar ~5GB de RAM
vectorizer = TfidfVectorizer(max_features=15000, ngram_range=(1,2))
X_tr_sparse = vectorizer.fit_transform(df['text'])
X_te_sparse = vectorizer.transform(df_test['text'])

print(f"Shape Treino: {X_tr_sparse.shape} | Shape Teste: {X_te_sparse.shape}")

## 3. Deep Learning: MMoE e Focal Loss (PyTorch)

Arquitetamos duas abordagens de Redes Neurais:
1. **Single-Task (Isolada)**: Uma rede separada para cada emoção.
2. **MMoE (Multi-gate Mixture-of-Experts)**: Uma mega rede onde 3 "Especialistas" (redes ocultas centrais) processam o texto, e Portões (Gates) dedicados aprendem o quanto devem escutar de cada especialista para prever uma emoção específica.

### Focal Loss Binária
Para resolver o colossal desbalanceamento do dataset (onde amostras felizes esmagam amostras irritadas), programamos a **Focal Loss**. Essa função de custo não apenas pesa as minorias, mas altera a derivada para focar exclusivamente nas "amostras difíceis" que o modelo está errando.

In [ ]:
# Habilitar GPU (CUDA) se disponivel
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Hardware Ativado: {device}")

class BinaryFocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super(BinaryFocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        bce_loss = nn.functional.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        probs = torch.sigmoid(inputs)
        pt = torch.where(targets == 1, probs, 1 - probs)
        
        focal_loss = ((1 - pt) ** self.gamma) * bce_loss
        
        if self.alpha is not None:
            alpha_t = torch.where(targets == 1, self.alpha, 1.0)
            focal_loss = focal_loss * alpha_t
            
        return focal_loss.mean()

class SingleTaskModel(nn.Module):
    def __init__(self, input_dim):
        super(SingleTaskModel, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 1)
        )
    def forward(self, x):
        return self.net(x).squeeze(1)

class Expert(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super(Expert, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3)
        )
    def forward(self, x):
        return self.net(x)

class Tower(nn.Module):
    def __init__(self, hidden_dim):
        super(Tower, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )
    def forward(self, x):
        return self.net(x).squeeze(1)

class MMoE_MultiTaskModel(nn.Module):
    def __init__(self, input_dim, num_experts=3, hidden_dim=256):
        super(MMoE_MultiTaskModel, self).__init__()
        self.num_experts = num_experts
        
        # Experts partilhados
        self.experts = nn.ModuleList([Expert(input_dim, hidden_dim) for _ in range(num_experts)])
        
        # Gates especificos para cada tarefa (Alegria, Tristeza, Raiva)
        self.gate_joy = nn.Linear(input_dim, num_experts)
        self.gate_sad = nn.Linear(input_dim, num_experts)
        self.gate_ang = nn.Linear(input_dim, num_experts)
        
        self.softmax = nn.Softmax(dim=1)
        
        # Torres especificas (camadas finais)
        self.tower_joy = Tower(hidden_dim)
        self.tower_sad = Tower(hidden_dim)
        self.tower_ang = Tower(hidden_dim)

    def forward(self, x):
        # 1. Processar todos os experts
        expert_outputs = torch.stack([expert(x) for expert in self.experts], dim=1) # [batch, num_experts, hidden_dim]
        
        # 2. Calcular os pesos dos gates
        w_joy = self.softmax(self.gate_joy(x)) # [batch, num_experts]
        w_sad = self.softmax(self.gate_sad(x))
        w_ang = self.softmax(self.gate_ang(x))
        
        # 3. Combinar as saidas dos experts pesadas pelos gates
        feat_joy = torch.sum(w_joy.unsqueeze(2) * expert_outputs, dim=1) # [batch, hidden_dim]
        feat_sad = torch.sum(w_sad.unsqueeze(2) * expert_outputs, dim=1)
        feat_ang = torch.sum(w_ang.unsqueeze(2) * expert_outputs, dim=1)
        
        # 4. Passar pelas torres finais
        out_joy = self.tower_joy(feat_joy)
        out_sad = self.tower_sad(feat_sad)
        out_ang = self.tower_ang(feat_ang)
        
        return out_joy, out_sad, out_ang

In [ ]:
# Conversao de matriz esparsa para Tensores densos por lote
X_train_t = torch.FloatTensor(X_tr_sparse.toarray()).to(device)
y_joy_train_t = torch.FloatTensor(df['y_joy'].values).to(device)
y_sad_train_t = torch.FloatTensor(df['y_sad'].values).to(device)
y_ang_train_t = torch.FloatTensor(df['y_ang'].values).to(device)

X_test_t = torch.FloatTensor(X_te_sparse.toarray()).to(device)

dataset_train = TensorDataset(X_train_t, y_joy_train_t, y_sad_train_t, y_ang_train_t)
loader_train = DataLoader(dataset_train, batch_size=256, shuffle=True)

# Pesos dinâmicos baseados na raridade
pos_w_joy = torch.tensor([(len(df) - df['y_joy'].sum()) / max(1, df['y_joy'].sum())]).to(device)
pos_w_sad = torch.tensor([(len(df) - df['y_sad'].sum()) / max(1, df['y_sad'].sum())]).to(device)
pos_w_ang = torch.tensor([(len(df) - df['y_ang'].sum()) / max(1, df['y_ang'].sum())]).to(device)

crit_joy = BinaryFocalLoss(alpha=pos_w_joy, gamma=2.0)
crit_sad = BinaryFocalLoss(alpha=pos_w_sad, gamma=2.0)
crit_ang = BinaryFocalLoss(alpha=pos_w_ang, gamma=2.0)

input_dim = 15000
epochs = 12

## Treinamento Deep Learning
Vamos treinar as Redes Single-Task e a Rede Multi-Task (MMoE) em paralelo por 12 Épocas usando o otimizador Adam.

In [ ]:
print("[Treinamento] Iniciando Redes Isoladas e MMoE...")
model_st_joy = SingleTaskModel(input_dim).to(device)
model_st_sad = SingleTaskModel(input_dim).to(device)
model_st_ang = SingleTaskModel(input_dim).to(device)
model_mt = MMoE_MultiTaskModel(input_dim).to(device)

opt_st_joy = optim.Adam(model_st_joy.parameters(), lr=0.001)
opt_st_sad = optim.Adam(model_st_sad.parameters(), lr=0.001)
opt_st_ang = optim.Adam(model_st_ang.parameters(), lr=0.001)
opt_mt = optim.Adam(model_mt.parameters(), lr=0.001)

sch_mt = optim.lr_scheduler.StepLR(opt_mt, step_size=5, gamma=0.5)

for ep in range(epochs):
    model_st_joy.train(); model_st_sad.train(); model_st_ang.train(); model_mt.train()
    
    for bx, by_j, by_s, by_a in loader_train:
        # Treino Single-Task
        opt_st_joy.zero_grad(); opt_st_sad.zero_grad(); opt_st_ang.zero_grad()
        loss_sj = crit_joy(model_st_joy(bx), by_j)
        loss_ss = crit_sad(model_st_sad(bx), by_s)
        loss_sa = crit_ang(model_st_ang(bx), by_a)
        loss_sj.backward(); loss_ss.backward(); loss_sa.backward()
        opt_st_joy.step(); opt_st_sad.step(); opt_st_ang.step()
        
        # Treino MMoE (Joint Loss)
        opt_mt.zero_grad()
        out_j, out_s, out_a = model_mt(bx)
        joint_loss = crit_joy(out_j, by_j) + crit_sad(out_s, by_s) + crit_ang(out_a, by_a)
        joint_loss.backward()
        opt_mt.step()
        
    sch_mt.step()
print("Redes Neurais Treinadas!")

## 4. O Duelo Final: Machine Learning Clássico

Agora convocamos os gigantes estatísticos da velha guarda.
Como explicado, a matriz TF-IDF de textos é 99% repleta de "zeros".
- As **Redes Neurais** sofrem pois multiplicam pesos por zero bilhões de vezes, desperdiçando cálculos.
- **Support Vector Machines (LinearSVC)** traçam retas geométricas perfeitas em dimensões super altas.
- **Árvores Aleatórias (Extra Trees)** sorteiam pedaços minúsculos de features a cada galho. Elas desviam do "oceano de zeros" incrivelmente rápido e extraem o sinal de modo cirúrgico.

Vamos treinar LinearSVC, LightGBM e ExtraTrees para disputarem a coroa contra a MMoE.

In [ ]:
print("[Treinamento] Iniciando Modelos Classicos...")
y_train_multi = df[['y_joy', 'y_sad', 'y_ang']].values
y_test_multi = df_test[['y_joy', 'y_sad', 'y_ang']].values

# SVM com C=4.0 (Hiperparametro ótimo descoberto via Grid Search prévio)
svc = MultiOutputClassifier(LinearSVC(C=4.0, class_weight='balanced', random_state=SEED))
svc.fit(X_tr_sparse, y_train_multi)
pred_svc = svc.predict(X_te_sparse)

# LightGBM (Gradient Boosting)
lgb = MultiOutputClassifier(LGBMClassifier(class_weight='balanced', random_state=SEED, n_jobs=-1, verbose=-1))
lgb.fit(X_tr_sparse, y_train_multi)
pred_lgb = lgb.predict(X_te_sparse)

# Extra Trees (Extremely Randomized Trees)
et = MultiOutputClassifier(ExtraTreesClassifier(n_estimators=100, class_weight='balanced', random_state=SEED, n_jobs=-1))
et.fit(X_tr_sparse, y_train_multi)
pred_et = et.predict(X_te_sparse)

## 5. Avaliação e Leaderboard Oficial (F1-Weighted)

A métrica *F1-Score Weighted* é a única balança matemática justa em *datasets* com forte desbalanceamento natural (como postagens de internet). Abaixo, os números definitivos deste projeto.

In [ ]:
model_st_joy.eval(); model_st_sad.eval(); model_st_ang.eval(); model_mt.eval()
with torch.no_grad():
    pred_st_joy = (torch.sigmoid(model_st_joy(X_test_t)) > 0.5).int().cpu().numpy()
    pred_st_sad = (torch.sigmoid(model_st_sad(X_test_t)) > 0.5).int().cpu().numpy()
    pred_st_ang = (torch.sigmoid(model_st_ang(X_test_t)) > 0.5).int().cpu().numpy()
    
    out_mt_j, out_mt_s, out_mt_a = model_mt(X_test_t)
    pred_mt_joy = (torch.sigmoid(out_mt_j) > 0.5).int().cpu().numpy()
    pred_mt_sad = (torch.sigmoid(out_mt_s) > 0.5).int().cpu().numpy()
    pred_mt_ang = (torch.sigmoid(out_mt_a) > 0.5).int().cpu().numpy()

def calc_f1(pred):
    f1_j = f1_score(y_test_multi[:, 0], pred[:, 0], average='weighted')
    f1_s = f1_score(y_test_multi[:, 1], pred[:, 1], average='weighted')
    f1_a = f1_score(y_test_multi[:, 2], pred[:, 2], average='weighted')
    return (f1_j + f1_s + f1_a) / 3

y_test_joy = y_test_multi[:, 0]
y_test_sad = y_test_multi[:, 1]
y_test_ang = y_test_multi[:, 2]

avg_st = ((f1_score(y_test_joy, pred_st_joy, average='weighted') +
           f1_score(y_test_sad, pred_st_sad, average='weighted') +
           f1_score(y_test_ang, pred_st_ang, average='weighted')) / 3)

avg_mt = ((f1_score(y_test_joy, pred_mt_joy, average='weighted') +
           f1_score(y_test_sad, pred_mt_sad, average='weighted') +
           f1_score(y_test_ang, pred_mt_ang, average='weighted')) / 3)

avg_svc = calc_f1(pred_svc)
avg_lgb = calc_f1(pred_lgb)
avg_et = calc_f1(pred_et)

print("="*60)
print("🏆 LEADERBOARD FINAL: F1-SCORE WEIGHTED (MULTILABEL) 🏆")
print("="*60)
print(f"[5] LightGBM Classifier       : {avg_lgb:.4f}")
print(f"[4] Redes Neurais Isoladas    : {avg_st:.4f}")
print(f"[3] Rede Multi-Tarefa (MMoE)  : {avg_mt:.4f}")
print(f"[2] LinearSVC (C=4.0)         : {avg_svc:.4f}")
print(f"[1] Extra Trees Classifier    : {avg_et:.4f} 🥇")
print("="*60)
